# 12 — Export Models to MLflow

Exports trained models (LayoutLM zone classifier, SBERT IFRS matcher)
to MLflow Model Registry for deployment. Registers model versions,
transitions to staging/production, and validates inference.

**Models**: layoutlm-zone-classifier, sbert-ifrs-matcher

In [ ]:
# Cell 1: Setup
!pip install -q mlflow transformers sentence-transformers torch PyYAML

import json, os
from pathlib import Path
import mlflow
import mlflow.pytorch
import mlflow.pyfunc
import torch
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

MODELS_DIR = Path(cfg['drive']['models_dir'])

# Configure MLflow
mlflow.set_tracking_uri(cfg['mlflow']['tracking_uri'])
print(f'MLflow URI: {cfg["mlflow"]["tracking_uri"]}')

In [ ]:
# Cell 2: Export LayoutLM Zone Classifier
from transformers import LayoutLMForTokenClassification, LayoutLMTokenizer

zone_model_path = MODELS_DIR / 'layoutlm-zone-classifier'
zcfg = cfg['zone_classifier']

if zone_model_path.exists():
    print(f'Loading zone model from: {zone_model_path}')
    zone_model = LayoutLMForTokenClassification.from_pretrained(str(zone_model_path))
    zone_tokenizer = LayoutLMTokenizer.from_pretrained(str(zone_model_path))
    
    # Register with MLflow
    mlflow.set_experiment(zcfg['mlflow_experiment'])
    with mlflow.start_run(run_name='export-zone-classifier'):
        # Log model
        mlflow.pytorch.log_model(
            zone_model,
            artifact_path='model',
            registered_model_name=zcfg['mlflow_model_name'],
        )
        
        # Log tokenizer as artifact
        tokenizer_dir = '/tmp/zone_tokenizer'
        zone_tokenizer.save_pretrained(tokenizer_dir)
        mlflow.log_artifacts(tokenizer_dir, 'tokenizer')
        
        # Log config
        mlflow.log_params(zcfg)
        
        run_id = mlflow.active_run().info.run_id
        print(f'Zone model registered: run_id={run_id}')
else:
    print(f'Zone model not found at {zone_model_path} — run notebook 07 first')

In [ ]:
# Cell 3: Export SBERT IFRS Matcher
from sentence_transformers import SentenceTransformer

sbert_model_path = MODELS_DIR / 'sbert-ifrs-finetuned'
scfg = cfg['sbert']

if sbert_model_path.exists():
    print(f'Loading SBERT model from: {sbert_model_path}')
    sbert_model = SentenceTransformer(str(sbert_model_path))
    
    mlflow.set_experiment(scfg['mlflow_experiment'])
    with mlflow.start_run(run_name='export-sbert-ifrs'):
        # Log as PyFunc for inference
        class SBERTWrapper(mlflow.pyfunc.PythonModel):
            def __init__(self, model_path):
                self.model_path = model_path
            
            def load_context(self, context):
                self.model = SentenceTransformer(self.model_path)
            
            def predict(self, context, model_input):
                texts = model_input['text'].tolist()
                embeddings = self.model.encode(texts)
                return embeddings.tolist()
        
        mlflow.pyfunc.log_model(
            artifact_path='model',
            python_model=SBERTWrapper(str(sbert_model_path)),
            registered_model_name=scfg['mlflow_model_name'],
            pip_requirements=['sentence-transformers', 'torch'],
        )
        
        mlflow.log_params(scfg)
        
        # Log baseline vs finetuned metrics if available
        baseline_file = Path(cfg['drive']['data_dir']) / 'sbert_baseline_results.json'
        if baseline_file.exists():
            with open(baseline_file) as f:
                baseline = json.load(f)
            for k, v in baseline.items():
                mlflow.log_metric(f'baseline_{k}', v)
        
        run_id = mlflow.active_run().info.run_id
        print(f'SBERT model registered: run_id={run_id}')
else:
    print(f'SBERT model not found at {sbert_model_path} — run notebook 09 first')

In [ ]:
# Cell 4: Transition models to Staging
from mlflow.tracking import MlflowClient

client = MlflowClient()

for model_name in [zcfg['mlflow_model_name'], scfg['mlflow_model_name']]:
    try:
        latest = client.get_latest_versions(model_name, stages=['None'])
        if latest:
            version = latest[0].version
            client.transition_model_version_stage(
                name=model_name,
                version=version,
                stage='Staging',
            )
            print(f'{model_name} v{version} → Staging ✅')
    except Exception as e:
        print(f'{model_name}: {e}')

In [ ]:
# Cell 5: Validate exported models — test inference
print('=== Validation: Zone Classifier ===')
if zone_model_path.exists():
    test_input = zone_tokenizer(
        ['Revenue', 'Total', 'Assets'],
        is_split_into_words=True,
        return_tensors='pt',
        padding=True,
        max_length=512,
        truncation=True,
    )
    test_input['bbox'] = torch.zeros(1, test_input['input_ids'].shape[1], 4, dtype=torch.long)
    with torch.no_grad():
        output = zone_model(**test_input)
    print(f'  Output shape: {output.logits.shape}')
    print(f'  Predicted classes: {output.logits.argmax(dim=-1).squeeze().tolist()}')

print('\n=== Validation: SBERT IFRS Matcher ===')
if sbert_model_path.exists():
    test_texts = ['Revenue', 'Total Assets', 'Net Income', 'Cash Flow from Operations']
    embeddings = sbert_model.encode(test_texts)
    print(f'  Embedding shape: {embeddings.shape}')
    from sentence_transformers import util as st_util
    sims = st_util.cos_sim(embeddings, embeddings)
    print(f'  Self-similarity matrix diagonal: {sims.diagonal().tolist()}')

print('\n✅ Export complete!')